# Model and Hyperparameter Search

## Model Selection
We first performed model selection using 3-fold stratified cross-validation on a `TF-IDF` unigram representation, evaluating several candidate models with `quadratic weighted kappa (QWK)` as the primary metric. Across datasets, linear models consistently outperformed other approaches, with `LinearSVC` and `LogisticRegression` achieving nearly identical performance. Given this parity, `LogisticRegression` was selected due to its simpler optimization and probabilistic interpretability. 

## Hyperparameter Tuning
In a second step, we conducted hyperparameter tuning on `LogisticRegression`, exploring different regularization strengths (`C`), penalties (`l1`, `l2`), and solvers (`lbfgs`, `saga`). While the best configuration for the largest dataset (`caes`) used the `saga` solver with stronger regularization (`C=10`), the smaller datasets favored `lbfgs` with `C=1`. Considering overall performance, stability, and simplicity, we selected a unified configuration using `LogisticRegression` with `l2` regularization (`solver="lbfgs"`, `C=1.0`) as the final model.

In [1]:
import os
import numpy as np
import pandas as pd

from nltk.corpus import stopwords

from sklearn.base import clone
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.linear_model import LogisticRegression, SGDClassifier, Ridge, LinearRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import accuracy_score, f1_score, cohen_kappa_score

## Data

In [2]:
outputfiles = ["caes_feats", "hablacultura_feats", "kwiziq_feats"]

dfs = []

for file in outputfiles:
    df = pd.read_csv(os.path.join("data", file + ".csv"))
    
    # add dataset column (remove "_feats")
    df["dataset"] = file.replace("_feats", "")
    
    dfs.append(df)

# concatenate all
data = pd.concat(dfs, ignore_index=True)
data.groupby("dataset")["cefr"].value_counts()

dataset       cefr
caes          A2      5708
              A1      5553
              B1      4489
              B2      3100
              C1      1786
hablacultura  B2       273
              B1       245
              C1        99
              A2        33
kwiziq        B2        59
              B1        54
              C1        39
              A2        33
              A1        21
Name: count, dtype: int64

## Model Selection
3-fold CV across datasets

In [3]:
# ====== GENERAL VARIABLES ======
target = "cefr"
text_col = "lemmas"

spanish_stopwords = stopwords.words("spanish")
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

# ====== CEFR ORDERED MAPPING ======
cefr_order = ["A1", "A2", "B1", "B2", "C1"]
cefr_to_int = {c: i for i, c in enumerate(cefr_order)}
int_to_cefr = {i: c for c, i in cefr_to_int.items()}

print("CEFR mapping:", cefr_to_int)

# ====== MODELS ======
candidate_models = {
    "tfidf_logreg": Pipeline([
        ("tfidf", TfidfVectorizer(
            stop_words=spanish_stopwords,
            ngram_range=(1, 1)
        )),
        ("clf", LogisticRegression(
            max_iter=2000,
            class_weight="balanced"
        ))
    ]),
    "tfidf_linear_svc": Pipeline([
        ("tfidf", TfidfVectorizer(
            stop_words=spanish_stopwords,
            ngram_range=(1, 1)
        )),
        ("clf", LinearSVC(
            class_weight="balanced"
        ))
    ]),
        "tfidf_linear_reg": Pipeline([
        ("tfidf", TfidfVectorizer(
            stop_words=spanish_stopwords,
            ngram_range=(1, 1)
        )),
        ("clf", LinearRegression())
    ]),
    "tfidf_sgd_hinge": Pipeline([
        ("tfidf", TfidfVectorizer(
            stop_words=spanish_stopwords,
            ngram_range=(1, 1)
        )),
        ("clf", SGDClassifier(
            loss="hinge",
            class_weight="balanced",
            random_state=42
        ))
    ]),
    "tfidf_multinomial_nb": Pipeline([
        ("tfidf", TfidfVectorizer(
            stop_words=spanish_stopwords,
            ngram_range=(1, 1)
        )),
        ("clf", MultinomialNB())
    ]),
    "tfidf_ridge": Pipeline([
        ("tfidf", TfidfVectorizer(
            stop_words=spanish_stopwords,
            ngram_range=(1, 1)
        )),
        ("clf", Ridge(alpha=1.0))
    ]),
    "tfidf_decision_tree": Pipeline([
        ("tfidf", TfidfVectorizer(
            stop_words=spanish_stopwords,
            ngram_range=(1, 1)
        )),
        ("clf", DecisionTreeClassifier(
            class_weight="balanced",
            random_state=42
        ))
    ]),
    "tfidf_random_forest": Pipeline([
        ("tfidf", TfidfVectorizer(
            stop_words=spanish_stopwords,
            ngram_range=(1, 1)
        )),
        ("clf", RandomForestClassifier(
            n_estimators=300,
            class_weight="balanced_subsample",
            random_state=42,
            n_jobs=-1
        ))
    ]),
}


# ====== MODEL SELECTION ======
rows = []

for ds_name, group in data.groupby("dataset"):
    print(f"\n=== DATASET: {ds_name} ===")

    X = group[text_col].fillna("").astype(str).values
    y_str = group[target].astype(str).str.upper().values
    y = np.array([cefr_to_int[v] for v in y_str])

    print("Class distribution:")
    print(pd.Series(y_str).value_counts().sort_index())

    for model_name, model in candidate_models.items():
        print(f"\n--- Model: {model_name} ---")

        qwk_list, acc_list, f1m_list = [], [], []

        for fold_id, (tr_idx, te_idx) in enumerate(cv.split(X, y), start=1):
            X_tr, X_te = X[tr_idx], X[te_idx]
            y_tr, y_te = y[tr_idx], y[te_idx]

            clf = clone(model)
            clf.fit(X_tr, y_tr)
            y_pred = clf.predict(X_te)

            # regression output -> ordinal labels
            if model_name == "tfidf_ridge":
                y_pred = np.round(y_pred)
                y_pred = np.clip(y_pred, 0, len(cefr_order) - 1)

            y_pred = y_pred.astype(int)

            qwk = cohen_kappa_score(y_te, y_pred, weights="quadratic")
            acc = accuracy_score(y_te, y_pred)
            f1m = f1_score(y_te, y_pred, average="macro")

            qwk_list.append(qwk)
            acc_list.append(acc)
            f1m_list.append(f1m)

            print(
                f"Fold {fold_id}: "
                f"QWK={qwk:.4f} | acc={acc:.4f} | f1_macro={f1m:.4f}"
            )

        row = {
            "dataset": ds_name,
            "model": model_name,
            "qwk_mean": float(np.mean(qwk_list)),
            "qwk_std": float(np.std(qwk_list)),
            "acc_mean": float(np.mean(acc_list)),
            "acc_std": float(np.std(acc_list)),
            "f1_macro_mean": float(np.mean(f1m_list)),
            "f1_macro_std": float(np.std(f1m_list)),
        }
        rows.append(row)

        print(
            f"Mean: QWK={row['qwk_mean']:.4f} ± {row['qwk_std']:.4f} | "
            f"acc={row['acc_mean']:.4f} ± {row['acc_std']:.4f} | "
            f"f1_macro={row['f1_macro_mean']:.4f} ± {row['f1_macro_std']:.4f}"
        )

# ====== SUMMARY ======
results = pd.DataFrame(rows).sort_values(
    by=["dataset", "qwk_mean"],
    ascending=[True, False]
)

print("\n=== MODEL SELECTION SUMMARY (sorted by QWK) ===")
print(results)

best_per_dataset = results.groupby("dataset", as_index=False).first()

print("\n=== BEST MODEL PER DATASET (by QWK) ===")
print(best_per_dataset[["dataset", "model", "qwk_mean", "qwk_std"]])

overall = (
    results.groupby("model", as_index=False)
    .agg(
        qwk_mean=("qwk_mean", "mean"),
        qwk_std=("qwk_mean", "std"),
        acc_mean=("acc_mean", "mean"),
        f1_macro_mean=("f1_macro_mean", "mean"),
    )
    .sort_values("qwk_mean", ascending=False)
)

print("\n=== OVERALL MODEL RANKING ACROSS DATASETS ===")
print(overall)

best_overall_model = overall.iloc[0]["model"]
#print(f"\nBest overall model to use further: {best_overall_model}")
best_overall_model

overall.to_csv("model_selection.csv", index=False)

CEFR mapping: {'A1': 0, 'A2': 1, 'B1': 2, 'B2': 3, 'C1': 4}

=== DATASET: caes ===
Class distribution:
A1    5553
A2    5708
B1    4489
B2    3100
C1    1786
Name: count, dtype: int64

--- Model: tfidf_logreg ---
Fold 1: QWK=0.9835 | acc=0.9769 | f1_macro=0.9806
Fold 2: QWK=0.9869 | acc=0.9807 | f1_macro=0.9836
Fold 3: QWK=0.9871 | acc=0.9837 | f1_macro=0.9858
Mean: QWK=0.9858 ± 0.0016 | acc=0.9804 ± 0.0028 | f1_macro=0.9833 ± 0.0022

--- Model: tfidf_linear_svc ---
Fold 1: QWK=0.9889 | acc=0.9881 | f1_macro=0.9896
Fold 2: QWK=0.9912 | acc=0.9885 | f1_macro=0.9897
Fold 3: QWK=0.9914 | acc=0.9916 | f1_macro=0.9921
Mean: QWK=0.9905 ± 0.0011 | acc=0.9894 ± 0.0016 | f1_macro=0.9905 ± 0.0012

--- Model: tfidf_linear_reg ---
Fold 1: QWK=0.5212 | acc=0.3623 | f1_macro=0.0833
Fold 2: QWK=0.2795 | acc=0.2653 | f1_macro=0.0432
Fold 3: QWK=0.7077 | acc=0.4850 | f1_macro=0.1651
Mean: QWK=0.5028 ± 0.1753 | acc=0.3709 ± 0.0899 | f1_macro=0.0972 ± 0.0507

--- Model: tfidf_sgd_hinge ---
Fold 1: QWK=0.

## Hyperparameter Finetuning

In [4]:
import numpy as np
import pandas as pd

from nltk.corpus import stopwords

from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import cohen_kappa_score, make_scorer


# =========================================================
# SETTINGS
# =========================================================
target = "cefr"
text_col = "lemmas"   # or "text"

spanish_stopwords = stopwords.words("spanish")

cefr_order = ["A1", "A2", "B1", "B2", "C1"]
cefr_to_int = {label: idx for idx, label in enumerate(cefr_order)}

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
qwk_scorer = make_scorer(cohen_kappa_score, weights="quadratic")


# =========================================================
# BASE PIPELINE
# =========================================================
pipe = Pipeline([
    ("tfidf", TfidfVectorizer(
        stop_words=spanish_stopwords,
        ngram_range=(1, 1)
    )),
    ("clf", LogisticRegression(
        max_iter=3000,
        class_weight="balanced"
    ))
])


# =========================================================
# PARAMETER GRID: regularization + C + solver
# =========================================================
param_grid = [
    {
        "clf__solver": ["lbfgs"],
        "clf__penalty": ["l2"],
        "clf__C": [0.01, 0.1, 1.0, 10.0, 100.0],
    },
    {
        "clf__solver": ["saga"],
        "clf__penalty": ["l1", "l2"],
        "clf__C": [0.01, 0.1, 1.0, 10.0],
    },
]


# =========================================================
# TUNING LOOP
# =========================================================
all_rows = []

for ds_name, group in data.groupby("dataset"):
    print(f"\n{'=' * 70}")
    print(f"TUNING DATASET: {ds_name}")
    print(f"{'=' * 70}")

    X = group[text_col].fillna("").astype(str).values
    y_str = group[target].astype(str).str.upper().values
    y = np.array([cefr_to_int[label] for label in y_str], dtype=int)

    search = GridSearchCV(
        estimator=pipe,
        param_grid=param_grid,
        scoring=qwk_scorer,
        cv=cv,
        n_jobs=-1,
        refit=True,
        verbose=1,
        return_train_score=False
    )

    search.fit(X, y)

    print(f"Best QWK: {search.best_score_:.4f}")
    print(f"Best params: {search.best_params_}")

    # collect ALL rows for this dataset
    cv_df = pd.DataFrame(search.cv_results_)

    keep_cols = [
        "mean_test_score",
        "std_test_score",
        "rank_test_score",
        "param_clf__solver",
        "param_clf__penalty",
        "param_clf__C",
    ]

    # keep split scores too, if they exist
    split_cols = [c for c in cv_df.columns if c.startswith("split") and c.endswith("_test_score")]
    keep_cols.extend(split_cols)

    cv_df = cv_df[keep_cols].copy()
    cv_df["dataset"] = ds_name

    cv_df = cv_df.rename(columns={
        "mean_test_score": "qwk_mean_cv",
        "std_test_score": "qwk_std_cv",
        "rank_test_score": "rank",
        "param_clf__solver": "solver",
        "param_clf__penalty": "penalty",
        "param_clf__C": "C",
    })

    all_rows.append(cv_df)

# =========================================================
# ALL RESULTS
# =========================================================
all_results = pd.concat(all_rows, ignore_index=True)

# sort globally by dataset, then by qwk descending
all_results = all_results.sort_values(
    by=["dataset", "qwk_mean_cv"],
    ascending=[True, False]
).reset_index(drop=True)

# save ALL results to csv
all_results.to_csv("gridsearch_all_results.csv", index=False)

print("\n=== ALL GRID SEARCH RESULTS ===")
print(all_results)

# =========================================================
# BEST RESULT PER DATASET
# =========================================================
best_results = (
    all_results
    .sort_values(by=["dataset", "qwk_mean_cv"], ascending=[True, False])
    .groupby("dataset", as_index=False)
    .first()
    .sort_values(by="qwk_mean_cv", ascending=False)   # sort datasets by best qwk
    .reset_index(drop=True)
)

print("\n=== BEST LOGISTIC REGRESSION SETTINGS PER DATASET ===")
print(best_results)

# save BEST results too
best_results.to_csv("gridsearch.csv", index=False)


TUNING DATASET: caes
Fitting 3 folds for each of 13 candidates, totalling 39 fits
Best QWK: 0.9893
Best params: {'clf__C': 10.0, 'clf__penalty': 'l2', 'clf__solver': 'saga'}

TUNING DATASET: hablacultura
Fitting 3 folds for each of 13 candidates, totalling 39 fits


/home/dimm/miniconda3/envs/py310/lib/python3.10/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/home/dimm/miniconda3/envs/py310/lib/python3.10/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/home/dimm/miniconda3/envs/py310/lib/python3.10/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/home/dimm/miniconda3/envs/py310/lib/python3.10/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/home/dimm/miniconda3/envs/py310/lib/python3.10/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/home/dimm/mini

Best QWK: 0.6512
Best params: {'clf__C': 1.0, 'clf__penalty': 'l2', 'clf__solver': 'lbfgs'}

TUNING DATASET: kwiziq
Fitting 3 folds for each of 13 candidates, totalling 39 fits
Best QWK: 0.4657
Best params: {'clf__C': 1.0, 'clf__penalty': 'l2', 'clf__solver': 'lbfgs'}

=== ALL GRID SEARCH RESULTS ===
    qwk_mean_cv  qwk_std_cv  rank solver penalty       C  split0_test_score  \
0      0.989278    0.001429     1   saga      l2   10.00           0.987295   
1      0.989023    0.001544     2  lbfgs      l2   10.00           0.986930   
2      0.988402    0.001048     3  lbfgs      l2  100.00           0.986926   
3      0.986094    0.001863     4   saga      l2    1.00           0.983544   
4      0.985839    0.001624     5  lbfgs      l2    1.00           0.983546   
5      0.981451    0.000896     6   saga      l1   10.00           0.981434   
6      0.970931    0.000739     7   saga      l2    0.10           0.971004   
7      0.970728    0.000983     8  lbfgs      l2    0.10          